In [25]:
from pyspark.sql import SparkSession,functions as F
from pyspark.sql.types import (
    StructType,StructField,
    IntegerType,StringType,
    TimestampType,DecimalType
)
from pathlib import Path
from pyspark.sql.window import Window

spark = SparkSession.builder.\
appName("FestiveDataDetective").\
master("local[*]").\
getOrCreate()

In [ ]:
#loading csv files with no schemainference to understand what the files actually contain
#So that can be inspected and to define the explicit schemas for all datasets
DATA_DIR = Path('Data/raw')

files = {
    'attendees':"attendees.csv",
    'food_orders':"food_orders.csv",
    "food_stalls":"food_stalls.csv",
    "incidents":"incidents.csv",
    "performances":"performances.csv",
    "stages":"stages.csv",
    "ticket_scans":"ticket_scans.csv"
}
raw = {}

for name,filename in files.items():
    filepath = DATA_DIR / filename

    raw[name] = (
        spark.read.
        option('header',True)
        .option('inferSchema',True)
        .option('mode','PERMISSIVE')
        .csv(str(DATA_DIR / filename))
    )
print("Successfully read",len(raw),"csv files")

Successfully read 7 csv files


In [32]:
#displaying row count,column details,5 records and understading the schema

for name, df in raw.items():
    print()
    print("=" * 50)
    print()
    print(name.upper())
    print("Number of rows: ",df.count())
    print(f"columns({len(df.columns)}): {df.columns}")
    print("-"*50)
    df.printSchema()
    df.show(5,truncate=False)



ATTENDEES
Number of rows:  1000
columns(4): ['attendee_id', 'ticket_type', 'age_group', 'home_city']
--------------------------------------------------
root
 |-- attendee_id: string (nullable = true)
 |-- ticket_type: string (nullable = true)
 |-- age_group: string (nullable = true)
 |-- home_city: string (nullable = true)

+-----------+-----------+---------+---------+
|attendee_id|ticket_type|age_group|home_city|
+-----------+-----------+---------+---------+
|A00001     |VIP        |25-34    |Bengaluru|
|A00002     |Student    |18-24    |Mumbai   |
|A00003     |general    |25-34    |Hyderabad|
|A00004     |General    |NULL     |Hyderabad|
|A00005     |VIP        |18-24    |Hyderabad|
+-----------+-----------+---------+---------+
only showing top 5 rows


FOOD_ORDERS
Number of rows:  1602
columns(6): ['order_id', 'attendee_id', 'stall_id', 'order_time', 'amount', 'order_status']
--------------------------------------------------
root
 |-- order_id: string (nullable = true)
 |-- atten

In [35]:
#defining explicit schemas for given datasets
schemas ={
    "attendees":StructType([
        StructField("attendee_id",StringType(),True),
        StructField("ticket_type",StringType(),True),
        StructField("age_group",StringType(),True),
        StructField("homecity",StringType(),True)
    ]),
    "food_orders":StructType([
        StructField("order_id",StringType(),True),
        StructField("attendee_id",StringType(),True),
        StructField("stall_id",StringType(),True),
        StructField("order_time",StringType(),True),
        StructField("amount",DecimalType(10,2),True),
        StructField("order_status",StringType(),True)
    ]),
    "food_stalls":StructType([
        StructField("stall_id",StringType(),True),
        StructField('stall_name',StringType(),True),
        StructField('cuisine',StringType(),True),
        StructField("stage_id",StringType(),True)
    ]),
    'incidents':StructType([
        StructField("incident_id",StringType(),True),
        StructField("stage_id",StringType(),True),
        StructField("incident_time",TimestampType(),True),
        StructField("incident_type",StringType(),True),
        StructField("severity",StringType(),True)
    ]),
    "performances":StructType([
        StructField("performance_id",StringType(),True),
        StructField("artist_name",StringType(),True),
        StructField('stage_id',StringType(),True),
        StructField('start_time',TimestampType(),True),
        StructField('end_time',TimestampType(),True),
        StructField('genre',StringType(),True)
    ]),
    "stages":StructType([
        StructField("stage_id",StringType(),True),
        StructField("stage_name",StringType(),True),
        StructField("capacity",StringType(),True),
        StructField('zone',StringType(),True)
    ]),
    "ticket_scans":StructType([
        StructField("scan_id",StringType(),True),
        StructField("attendee_id",StringType(),True),
        StructField("stage_id",StringType(),True),
        StructField("scan_time",TimestampType(),True),
        StructField("scan_status",StringType(),True)
    ])
}

assert set(schemas) == set(files)
print("Schemas defined: ",len(schemas))

Schemas defined:  7


In [38]:
#reading datasets using defined schemas

typed = {}

for name,filename in files.items():
    typed[name] = (
        spark.read
        .schema(schemas[name])
        .option("header",True)
        .option("mode",'PERMISSIVE')
        .option('timestampFormat','yyyy-MM-dd HH:mm:ss')
        .csv(str(DATA_DIR/filename))
        
    )

print("Typed Datasets loaded: ",len(typed))

# Optional aliases for dataset-specific analysis

attendees_df = typed["attendees"]
ticket_scans_df = typed["ticket_scans"]
food_orders_df = typed["food_orders"]
stages_df = typed["stages"]
performances_df = typed["performances"]
food_stalls_df = typed["food_stalls"]
incidents_df = typed["incidents"]

Typed Datasets loaded:  7


In [41]:
#finding total row counts, missing values of each column for each dataset
missing_value_reports = {}

for name,df in typed.items():
    print(f'\n{'=' * 60}')
    print(name.upper())

    expressions = [F.count("*").alias("row_count")]

    for column in df.columns:
        expressions.append(
            F.sum(
                    F.when(
                        F.col(column).isNull() |
                        (F.trim(F.col(column).cast('string'))==""), 1
                    ).otherwise(0)
                ).alias(f'{column}_missing')
        )
    report = df.agg(*expressions)
    missing_value_reports[name] = report
    report.show(truncate = False)



ATTENDEES
+---------+-------------------+-------------------+-----------------+----------------+
|row_count|attendee_id_missing|ticket_type_missing|age_group_missing|homecity_missing|
+---------+-------------------+-------------------+-----------------+----------------+
|1000     |0                  |0                  |154              |266             |
+---------+-------------------+-------------------+-----------------+----------------+


FOOD_ORDERS
+---------+----------------+-------------------+----------------+------------------+--------------+--------------------+
|row_count|order_id_missing|attendee_id_missing|stall_id_missing|order_time_missing|amount_missing|order_status_missing|
+---------+----------------+-------------------+----------------+------------------+--------------+--------------------+
|1602     |0               |0                  |0               |0                 |39            |0                   |
+---------+----------------+-------------------+--------

In [45]:
#identifying the duplicate key groups

primary_keys = {
    "attendees":'attendee_id',
    "food_orders":"order_id",
    "food_stalls":"stall_id",
    "incidents":"incident_id",
    "performances":"performance_id",
    "stages":"stage_id",
    "ticket_scans":"scan_id"
}
duplicate_key_reports = {}
for name, key in primary_keys.items():
    duplicate_groups = (
        typed[name].groupBy(key)
        .count()
        .filter(
            F.col(key).isNotNull() & 
            (F.col("count") > 1)
        )
    )
    duplicate_key_reports[name] = duplicate_groups
    print(f'\n{'='*50}')
    print(f'Dataset: {name.upper()}')
    print(f'Primary key: {key}')
    duplicate_groups.show(30,truncate=False)
    print(
        "Number of duplicate key groups:",
        duplicate_groups.count()
    )


Dataset: ATTENDEES
Primary key: attendee_id
+-----------+-----+
|attendee_id|count|
+-----------+-----+
+-----------+-----+

Number of duplicate key groups: 0

Dataset: FOOD_ORDERS
Primary key: order_id
+--------+-----+
|order_id|count|
+--------+-----+
|O00006  |2    |
|O00056  |2    |
+--------+-----+

Number of duplicate key groups: 2

Dataset: FOOD_STALLS
Primary key: stall_id
+--------+-----+
|stall_id|count|
+--------+-----+
+--------+-----+

Number of duplicate key groups: 0

Dataset: INCIDENTS
Primary key: incident_id
+-----------+-----+
|incident_id|count|
+-----------+-----+
+-----------+-----+

Number of duplicate key groups: 0

Dataset: PERFORMANCES
Primary key: performance_id
+--------------+-----+
|performance_id|count|
+--------------+-----+
+--------------+-----+

Number of duplicate key groups: 0

Dataset: STAGES
Primary key: stage_id
+--------+-----+
|stage_id|count|
+--------+-----+
+--------+-----+

Number of duplicate key groups: 0

Dataset: TICKET_SCANS
Primary k

In [ ]:

category_columns = {
    "attendees": ["ticket_type", "age_group", "home_city"],
    "food_orders": ["order_status"],
    "food_stalls": ["cuisine"],
    "incidents": ["incident_type", "severity"],
    "performances": ["genre"],
    "stages": ["zone"],
    "ticket_scans": ["scan_status"],
}

for name, columns in category_columns.items():
    print(f"\n{'=' * 55}")
    print(name.upper())

    for column in columns:
        print(f"\nValues in {column}:")

        (
            typed[name]
            .groupBy(F.lower(F.trim(F.col(column).cast("string"))).alias("normalized_value"))
            .count()
            .orderBy(F.desc("count"))
            .show(30, truncate=False)
        )





ATTENDEES

Values in ticket_type:
+----------------+-----+
|normalized_value|count|
+----------------+-----+
|general         |322  |
|vip             |321  |
|student         |186  |
|day pass        |171  |
+----------------+-----+


Values in age_group:
+----------------+-----+
|normalized_value|count|
+----------------+-----+
|unknown         |182  |
|45+             |182  |
|25-34           |161  |
|18-24           |161  |
|35-44           |160  |
|NULL            |154  |
+----------------+-----+


Values in home_city:
+----------------+-----+
|normalized_value|count|
+----------------+-----+
|NULL            |266  |
|delhi           |130  |
|mumbai          |123  |
|pune            |122  |
|chennai         |121  |
|bengaluru       |121  |
|hyderabad       |117  |
+----------------+-----+


FOOD_ORDERS

Values in order_status:
+----------------+-----+
|normalized_value|count|
+----------------+-----+
|completed       |607  |
|pending         |374  |
|cancelled       |317  |
|refu

In [9]:

print("FOOD ORDERS — amount distribution")
typed["food_orders"].select(
    F.min("amount").alias("min_amount"),
    F.max("amount").alias("max_amount"),
    F.sum(F.when(F.col("amount") < 0, 1).otherwise(0)).alias("negative_amounts"),
    F.sum(F.when(F.col("amount") == 0, 1).otherwise(0)).alias("zero_amounts"),
).show()

print("STAGES — capacity distribution")
typed["stages"].select(
    F.min("capacity").alias("min_capacity"),
    F.max("capacity").alias("max_capacity"),
    F.sum(F.when(F.col("capacity") <= 0, 1).otherwise(0)).alias("nonpositive_capacities"),
).show()

print("PERFORMANCES — invalid time ranges")
typed["performances"].filter(
    F.col("start_time").isNotNull()
    & F.col("end_time").isNotNull()
    & (F.col("end_time") <= F.col("start_time"))
).show(truncate=False)


FOOD ORDERS — amount distribution
+----------+----------+----------------+------------+
|min_amount|max_amount|negative_amounts|zero_amounts|
+----------+----------+----------------+------------+
|    -12.50|     54.97|               1|           0|
+----------+----------+----------------+------------+

STAGES — capacity distribution
+------------+------------+----------------------+
|min_capacity|max_capacity|nonpositive_capacities|
+------------+------------+----------------------+
|        1200|        8500|                     0|
+------------+------------+----------------------+

PERFORMANCES — invalid time ranges
+--------------+-----------+--------+----------+--------+-----+
|performance_id|artist_name|stage_id|start_time|end_time|genre|
+--------------+-----------+--------+----------+--------+-----+
+--------------+-----------+--------+----------+--------+-----+



In [10]:
# 1. Inspect repeated food order IDs
print("DUPLICATE FOOD ORDERS")

duplicate_order_ids = (
    typed["food_orders"]
    .groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
    .select("order_id")
)

(
    typed["food_orders"]
    .join(duplicate_order_ids, on="order_id", how="inner")
    .orderBy("order_id")
    .show(truncate=False)
)

# 2. Inspect repeated scan IDs
print("DUPLICATE TICKET SCANS")

duplicate_scan_ids = (
    typed["ticket_scans"]
    .groupBy("scan_id")
    .count()
    .filter(F.col("count") > 1)
    .select("scan_id")
)

(
    typed["ticket_scans"]
    .join(duplicate_scan_ids, on="scan_id", how="inner")
    .orderBy("scan_id")
    .show(truncate=False)
)

# 3. Inspect negative food amounts
print("NEGATIVE FOOD AMOUNTS")

(
    typed["food_orders"]
    .filter(F.col("amount") < 0)
    .show(truncate=False)
)


DUPLICATE FOOD ORDERS
+--------+-----------+--------+-------------------+------+------------+
|order_id|attendee_id|stall_id|order_time         |amount|order_status|
+--------+-----------+--------+-------------------+------+------------+
|O00006  |A00608     |FS02    |2026-09-13 20:35:00|43.07 |pending     |
|O00006  |A00608     |FS02    |2026-09-13 20:35:00|43.07 |pending     |
|O00056  |A00091     |FS05    |2026-09-11 18:56:00|20.87 |pending     |
|O00056  |A00091     |FS05    |2026-09-11 18:56:00|20.87 |pending     |
+--------+-----------+--------+-------------------+------+------------+

DUPLICATE TICKET SCANS
+--------+-----------+--------+-------------------+-----------+
|scan_id |attendee_id|stage_id|scan_time          |scan_status|
+--------+-----------+--------+-------------------+-----------+
|SC000016|A00002     |ST02    |2026-09-12 13:08:00|valid      |
|SC000016|A00002     |ST02    |2026-09-12 13:08:00|valid      |
|SC000029|A00522     |ST02    |2026-09-13 18:18:00|valid  

In [11]:

def show_orphan_records(
    source_df,
    source_key,
    target_df,
    target_key,
    relationship_name
):
    target_keys = (
        target_df
        .select(F.col(target_key).alias("_target_key"))
        .filter(F.col("_target_key").isNotNull())
        .distinct()
    )

    orphans = (
        source_df
        .filter(F.col(source_key).isNotNull())
        .join(
            target_keys,
            F.col(source_key) == F.col("_target_key"),
            "left_anti"
        )
    )

    print(f"\n{relationship_name}")
    print("Orphan rows:", orphans.count())
    orphans.show(10, truncate=False)


show_orphan_records(
    typed["food_orders"], "attendee_id",
    typed["attendees"], "attendee_id",
    "Food orders -> attendees"
)

show_orphan_records(
    typed["food_orders"], "stall_id",
    typed["food_stalls"], "stall_id",
    "Food orders -> food stalls"
)

show_orphan_records(
    typed["ticket_scans"], "attendee_id",
    typed["attendees"], "attendee_id",
    "Ticket scans -> attendees"
)

show_orphan_records(
    typed["ticket_scans"], "stage_id",
    typed["stages"], "stage_id",
    "Ticket scans -> stages"
)

show_orphan_records(
    typed["performances"], "stage_id",
    typed["stages"], "stage_id",
    "Performances -> stages"
)

show_orphan_records(
    typed["incidents"], "stage_id",
    typed["stages"], "stage_id",
    "Incidents -> stages"
)



Food orders -> attendees
Orphan rows: 0
+--------+-----------+--------+----------+------+------------+
|order_id|attendee_id|stall_id|order_time|amount|order_status|
+--------+-----------+--------+----------+------+------------+
+--------+-----------+--------+----------+------+------------+


Food orders -> food stalls
Orphan rows: 1
+--------+-----------+--------+-------------------+------+------------+
|order_id|attendee_id|stall_id|order_time         |amount|order_status|
+--------+-----------+--------+-------------------+------+------------+
|O00026  |A00118     |FS99    |2026-09-13 21:31:00|4.07  |cancelled   |
+--------+-----------+--------+-------------------+------+------------+


Ticket scans -> attendees
Orphan rows: 0
+-------+-----------+--------+---------+-----------+
|scan_id|attendee_id|stage_id|scan_time|scan_status|
+-------+-----------+--------+---------+-----------+
+-------+-----------+--------+---------+-----------+


Ticket scans -> stages
Orphan rows: 1
+-------

In [12]:

# Expected primary keys for each dataset
primary_keys = {
    "attendees": "attendee_id",
    "food_orders": "order_id",
    "food_stalls": "stall_id",
    "incidents": "incident_id",
    "performances": "performance_id",
    "stages": "stage_id",
    "ticket_scans": "scan_id",
}

# Records with duplicate primary keys
duplicate_keys = {}

for name, key in primary_keys.items():
    duplicate_keys[name] = (
        typed[name]
        .groupBy(key)
        .count()
        .filter(F.col(key).isNotNull() & (F.col("count") > 1))
        .select(key)
    )

# Reference keys
attendee_keys = typed["attendees"].select(
    F.col("attendee_id").alias("_ref_id")
).distinct()

stall_keys = typed["food_stalls"].select(
    F.col("stall_id").alias("_ref_id")
).distinct()

stage_keys = typed["stages"].select(
    F.col("stage_id").alias("_ref_id")
).distinct()


def unmatched_nonnull(df, column, reference_keys):
    """Rows whose non-null reference key doesn't exist in the lookup."""
    return (
        df.filter(F.col(column).isNotNull())
        .join(
            reference_keys,
            F.col(column) == F.col("_ref_id"),
            "left_anti"
        )
    )


# Define dataset-specific invalid-row rules
invalid_rules = {
    "attendees": lambda df: (
        F.col("attendee_id").isNull()
        | (F.trim("attendee_id") == "")
    ),

    "food_orders": lambda df: (
        F.col("order_id").isNull()
        | (F.trim("order_id") == "")
        | F.col("order_time").isNull()
        | F.col("amount").isNull()
        | (F.col("amount") < 0)
        | F.col("attendee_id").isNull()
        | F.col("stall_id").isNull()
    ),

    "food_stalls": lambda df: (
        F.col("stall_id").isNull()
        | (F.trim("stall_id") == "")
    ),

    "incidents": lambda df: (
        F.col("incident_id").isNull()
        | (F.trim("incident_id") == "")
        | F.col("stage_id").isNull()
        | F.col("incident_time").isNull()
        | (F.trim("incident_type") == "")
    ),

    "performances": lambda df: (
        F.col("performance_id").isNull()
        | (F.trim("performance_id") == "")
        | F.col("stage_id").isNull()
        | F.col("start_time").isNull()
        | F.col("end_time").isNull()
        | (F.col("end_time") <= F.col("start_time"))
        | (F.trim("artist_name") == "")
    ),

    "stages": lambda df: (
        F.col("stage_id").isNull()
        | (F.trim("stage_id") == "")
        | F.col("capacity").isNull()
        | (F.col("capacity") <= 0)
    ),

    "ticket_scans": lambda df: (
        F.col("scan_id").isNull()
        | (F.trim("scan_id") == "")
        | F.col("attendee_id").isNull()
        | F.col("stage_id").isNull()
        | F.col("scan_time").isNull()
    ),
}

# Reference-integrity rules
reference_rules = {
    "food_orders": [
        ("attendee_id", attendee_keys),
        ("stall_id", stall_keys),
    ],
    "ticket_scans": [
        ("attendee_id", attendee_keys),
        ("stage_id", stage_keys),
    ],
    "performances": [
        ("stage_id", stage_keys),
    ],
    "incidents": [
        ("stage_id", stage_keys),
    ],
}

clean = {}
quarantine = {}

for name, df in typed.items():
    # Attach duplicate and base validation flags
    base_invalid = invalid_rules[name](df)

    duplicate_condition = F.lit(False)
    for key in [primary_keys[name]]:
        duplicate_condition = duplicate_condition | (
            F.col(key).isNotNull()
            & F.col(key).isin(
                # Use a join instead of collecting all keys to Python.
                # The join-based duplicate logic is applied below.
                []
            )
        )

    # First quarantine duplicate key groups using left-semi join
    key = primary_keys[name]
    duplicate_rows = df.join(
        duplicate_keys[name], on=key, how="left_semi"
    )

    nonduplicate_rows = df.join(
        duplicate_keys[name], on=key, how="left_anti"
    )

    # Mark base-invalid records
    base_bad = nonduplicate_rows.filter(base_invalid)
    remaining = nonduplicate_rows.filter(~base_invalid)

    # Collect invalid rows for this dataset without dropping them silently
    invalid_parts = [
        duplicate_rows.withColumn(
            "quarantine_reason", F.lit("duplicate_primary_key")
        ),
        base_bad.withColumn(
            "quarantine_reason", F.lit("required_field_or_value_invalid")
        ),
    ]

    # Apply foreign-key rules one by one
    for fk_column, reference_df in reference_rules.get(name, []):
        bad_fk = unmatched_nonnull(
            remaining, fk_column, reference_df
        )
        if bad_fk.limit(1).count() > 0:
            invalid_parts.append(
                bad_fk.withColumn(
                    "quarantine_reason",
                    F.lit(f"unmatched_reference:{fk_column}")
                )
            )
        remaining = remaining.join(
            reference_df,
            F.col(fk_column) == F.col("_ref_id"),
            "left_semi"
        ).drop("_ref_id")

    # Union quarantine records with reason labels
    quarantine[name] = invalid_parts[0]
    for part in invalid_parts[1:]:
        quarantine[name] = quarantine[name].unionByName(
            part, allowMissingColumns=True
        )

    clean[name] = remaining

print("Clean and quarantine DataFrames created.")


Clean and quarantine DataFrames created.


In [13]:
for name in typed:
    print(f"\n{name.upper()}")
    print("Raw rows:", typed[name].count())
    print("Clean rows:", clean[name].count())
    print("Quarantined rows:", quarantine[name].count())

    quarantine[name].groupBy("quarantine_reason").count().show(
        truncate=False
    )


ATTENDEES
Raw rows: 1000
Clean rows: 1000
Quarantined rows: 0
+-----------------+-----+
|quarantine_reason|count|
+-----------------+-----+
+-----------------+-----+


FOOD_ORDERS
Raw rows: 1602
Clean rows: 1557
Quarantined rows: 45
+-------------------------------+-----+
|quarantine_reason              |count|
+-------------------------------+-----+
|duplicate_primary_key          |4    |
|required_field_or_value_invalid|40   |
|unmatched_reference:stall_id   |1    |
+-------------------------------+-----+


FOOD_STALLS
Raw rows: 6
Clean rows: 6
Quarantined rows: 0
+-----------------+-----+
|quarantine_reason|count|
+-----------------+-----+
+-----------------+-----+


INCIDENTS
Raw rows: 180
Clean rows: 178
Quarantined rows: 2
+-------------------------------+-----+
|quarantine_reason              |count|
+-------------------------------+-----+
|required_field_or_value_invalid|1    |
|unmatched_reference:stage_id   |1    |
+-------------------------------+-----+


PERFORMANCES
Raw r

In [14]:

primary_keys = {
    "attendees": "attendee_id",
    "food_orders": "order_id",
    "food_stalls": "stall_id",
    "incidents": "incident_id",
    "performances": "performance_id",
    "stages": "stage_id",
    "ticket_scans": "scan_id",
}

for name in typed:
    raw_count = typed[name].count()
    clean_count = clean[name].count()
    quarantine_count = quarantine[name].count()

    key = primary_keys[name]

    duplicate_groups = (
        clean[name]
        .groupBy(key)
        .count()
        .filter(F.col(key).isNotNull() & (F.col("count") > 1))
        .count()
    )

    print(
        f"{name:15} "
        f"raw={raw_count:4} "
        f"clean={clean_count:4} "
        f"quarantine={quarantine_count:3} "
        f"reconciles={raw_count == clean_count + quarantine_count} "
        f"duplicate_key_groups={duplicate_groups}"
    )

attendees       raw=1000 clean=1000 quarantine=  0 reconciles=True duplicate_key_groups=0
food_orders     raw=1602 clean=1557 quarantine= 45 reconciles=True duplicate_key_groups=0
food_stalls     raw=   6 clean=   6 quarantine=  0 reconciles=True duplicate_key_groups=0
incidents       raw= 180 clean= 178 quarantine=  2 reconciles=True duplicate_key_groups=0
performances    raw=  30 clean=  28 quarantine=  2 reconciles=True duplicate_key_groups=0
stages          raw=   5 clean=   5 quarantine=  0 reconciles=True duplicate_key_groups=0
ticket_scans    raw=2803 clean=2794 quarantine=  9 reconciles=True duplicate_key_groups=0


In [15]:

(
    quarantine["food_orders"]
    .filter(
        F.col("quarantine_reason")
        == "required_field_or_value_invalid"
    )
    .select(
        "order_id",
        "attendee_id",
        "stall_id",
        "order_time",
        "amount",
        "order_status",
        "quarantine_reason"
    )
    .orderBy("order_id")
    .show(50, truncate=False)
)

+--------+-----------+--------+-------------------+------+------------+-------------------------------+
|order_id|attendee_id|stall_id|order_time         |amount|order_status|quarantine_reason              |
+--------+-----------+--------+-------------------+------+------------+-------------------------------+
|O00004  |A00585     |FS03    |2026-09-11 12:28:00|NULL  |cancelled   |required_field_or_value_invalid|
|O00032  |A00455     |FS06    |2026-09-12 22:33:00|-12.50|Completed   |required_field_or_value_invalid|
|O00044  |A00582     |FS03    |2026-09-12 23:49:00|NULL  |Completed   |required_field_or_value_invalid|
|O00136  |A00708     |FS06    |2026-09-11 13:42:00|NULL  |pending     |required_field_or_value_invalid|
|O00215  |A00573     |FS03    |2026-09-13 16:24:00|NULL  |refunded    |required_field_or_value_invalid|
|O00243  |A00862     |FS06    |2026-09-11 15:38:00|NULL  |pending     |required_field_or_value_invalid|
|O00302  |A00044     |FS05    |2026-09-11 19:10:00|NULL  |cancel

In [16]:
(
    quarantine["food_orders"]
    .filter(F.col("quarantine_reason") == "duplicate_primary_key")
    .select("order_id", "attendee_id", "stall_id", "order_time", "amount", "order_status")
    .orderBy("order_id")
    .show(truncate=False)
)

+--------+-----------+--------+-------------------+------+------------+
|order_id|attendee_id|stall_id|order_time         |amount|order_status|
+--------+-----------+--------+-------------------+------+------------+
|O00006  |A00608     |FS02    |2026-09-13 20:35:00|43.07 |pending     |
|O00006  |A00608     |FS02    |2026-09-13 20:35:00|43.07 |pending     |
|O00056  |A00091     |FS05    |2026-09-11 18:56:00|20.87 |pending     |
|O00056  |A00091     |FS05    |2026-09-11 18:56:00|20.87 |pending     |
+--------+-----------+--------+-------------------+------+------------+



In [17]:


food_orders_raw = typed["food_orders"]

# Identify IDs that occur more than once
duplicate_ids = (
    food_orders_raw
    .groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
    .select("order_id")
)

# Check whether duplicate groups are identical across all columns
duplicate_records = food_orders_raw.join(
    duplicate_ids,
    on="order_id",
    how="inner"
)

duplicate_variants = (
    duplicate_records
    .dropDuplicates()
    .groupBy("order_id")
    .count()
)

duplicate_variants.show(truncate=False)

+--------+-----+
|order_id|count|
+--------+-----+
|O00006  |1    |
|O00056  |1    |
+--------+-----+



In [ ]:


food_orders_raw = typed["food_orders"]

order_window = (
    Window
    .partitionBy("order_id")
    .orderBy(
        F.col("order_time").desc_nulls_last(),
        F.col("amount").desc_nulls_last()
    )
)

food_orders_ranked = food_orders_raw.withColumn(
    "row_num",
    F.row_number().over(order_window)
)

food_orders_deduplicated = (
    food_orders_ranked
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

In [19]:
print("Raw rows:", food_orders_raw.count())
print("Deduplicated rows:", food_orders_deduplicated.count())

(
    food_orders_deduplicated
    .groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
    .show(truncate=False)
)

Raw rows: 1602
Deduplicated rows: 1600
+--------+-----+
|order_id|count|
+--------+-----+
+--------+-----+

